# Phase 1 Final Claim-Package Specification / Readiness Plan

Notebook fix marker: `phase1_final_claim_package_plan_v1_20260421`

Purpose: record the final Phase 1 claim-package contract after the governance readiness package. This notebook does **not** train models, execute controls, compute calibration, compute influence, compile final reporting, or open headline claims.

Scientific integrity rule: this notebook is an engineering plan/checklist. Missing final artifacts must remain missing/blocking; do not infer them from smoke metrics or fill them by hand.

## Technical Basis And Boundary

This notebook is grounded in the current repository contract:

- `docs/01_v55_doc_code_crosswalk.md`: durable scientific logic must live in `src/` and notebooks must only orchestrate CLI and inspect artifacts.
- `docs/02_colab_local_runbook.md`: Phase 1 headline claims require final comparator readiness plus executable controls, calibration, influence and reporting artifacts that pass locked thresholds.
- `configs/phase1/final_claim_package.json`: machine-readable final claim-package contract.
- `src/phase1/final_claim_package.py`: fail-closed plan runner. It records required final artifacts and blockers; it does not produce evidence.

Expected current result: `phase1_final_claim_package_plan_recorded`, with `claim_ready=false`, `headline_phase1_claim_open=false`, and blockers for final comparator/control/calibration/influence/reporting artifacts.

In [ ]:
# Cell 1 - Runtime setup, Drive mount, clone/pull repo, and unit tests.
# This cell intentionally runs tests before writing the final claim-package plan.

from pathlib import Path
import base64
import getpass
import subprocess

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
REPO_DIR = Path('/content/eeg-ds004752') if IN_COLAB else Path.cwd()
DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752') if IN_COLAB else Path.cwd()

def run(cmd, cwd=None, check=True, redact_token=True):
    display = list(map(str, cmd))
    if redact_token:
        display = ['<redacted-token-command>' if 'http.extraHeader=Authorization:' in item else item for item in display]
    print('$', ' '.join(display))
    result = subprocess.run(cmd, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
    return result

def github_auth_args():
    if not IN_COLAB:
        return []
    token = getpass.getpass('Nhập GitHub token cho repo private; Enter nếu runtime đã có quyền: ')
    if not token:
        return []
    header = 'Authorization: Basic ' + base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    return ['-c', f'http.extraHeader={header}']

GIT_AUTH_ARGS = github_auth_args()
if not REPO_DIR.exists():
    run(['git', *GIT_AUTH_ARGS, 'clone', REPO_URL, str(REPO_DIR)])
else:
    print(f'Repo exists: {REPO_DIR}')

run(['git', *GIT_AUTH_ARGS, 'pull', '--ff-only'], cwd=REPO_DIR)
run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)
unit_result = run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=REPO_DIR, check=False)
if unit_result.returncode != 0:
    print('Unit tests failed. Stop here; do not write final claim-package plan artifacts from an unclean codebase.')
    raise subprocess.CalledProcessError(unit_result.returncode, ['python', '-m', 'unittest', 'discover', '-s', 'tests'])
print('Unit tests passed; continuing to final claim-package plan checks.')

In [ ]:
# Cell 2 - Explicit source paths.
# Use the reviewed governance readiness run from notebook 15. Do not silently follow latest.txt for claim-affecting planning.

from pathlib import Path

PREREG_BUNDLE = DRIVE_ROOT / 'artifacts/prereg/20260418T161442014597Z/prereg_bundle.json'
GOVERNANCE_READINESS_RUN = DRIVE_ROOT / 'artifacts/phase1_governance_readiness/20260421T072655938750Z'
OUTPUT_ROOT = DRIVE_ROOT / 'artifacts/phase1_final_claim_package_plan'
PACKAGE_CONFIG = REPO_DIR / 'configs/phase1/final_claim_package.json'

EXPECTED_LOCKED_PREREG_IDENTITY_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'
EXPECTED_GOVERNANCE_STATUS = 'phase1_governance_readiness_blocked'
EXPECTED_SMOKE_REVIEWS = ['A2_A2b', 'A2c_CORAL', 'A2d_riemannian', 'A3_distillation', 'A4_privileged']

print('Prereg bundle:', PREREG_BUNDLE)
print('Governance readiness run:', GOVERNANCE_READINESS_RUN)
print('Package config:', PACKAGE_CONFIG)
print('Plan output root:', OUTPUT_ROOT)

In [ ]:
# Cell 3 - JSON/hash helpers used only for orchestration review.

import hashlib
import json
from pathlib import Path


def read_json(path: Path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    return json.loads(path.read_text(encoding='utf-8'))


def sha256_file(path: Path) -> str:
    path = Path(path)
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def latest_run(root: Path) -> Path:
    pointer = Path(root) / 'latest.txt'
    if not pointer.exists():
        raise FileNotFoundError(pointer)
    return Path(pointer.read_text(encoding='utf-8').strip())

In [ ]:
# Cell 4 - Validate prereg identity and governance readiness boundary.

bundle = read_json(PREREG_BUNDLE)
actual_prereg_hash = bundle.get('prereg_bundle_hash_sha256')
raw_prereg_file_sha256 = sha256_file(PREREG_BUNDLE)
print('Prereg status:', bundle.get('status'))
print('Locked prereg identity hash:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_file_sha256)
assert bundle.get('status') == 'locked', 'Prereg bundle must be locked.'
assert actual_prereg_hash == EXPECTED_LOCKED_PREREG_IDENTITY_HASH, 'Locked prereg identity hash mismatch.'

gov_summary = read_json(GOVERNANCE_READINESS_RUN / 'phase1_governance_readiness_summary.json')
gov_claim_state = read_json(GOVERNANCE_READINESS_RUN / 'phase1_claim_state.json')
print('Governance status:', gov_summary.get('status'))
print('Governance claim_ready:', gov_summary.get('claim_ready'))
print('Completed non-claim smoke reviews:', gov_summary.get('completed_non_claim_smoke_reviews'))
assert gov_summary.get('status') == EXPECTED_GOVERNANCE_STATUS
assert gov_summary.get('claim_ready') is False
assert gov_summary.get('headline_phase1_claim_open') is False
assert gov_claim_state.get('full_phase1_claim_bearing_run_allowed') is False
for item in EXPECTED_SMOKE_REVIEWS:
    assert item in gov_summary.get('completed_non_claim_smoke_reviews', []), f'Missing smoke review: {item}'

In [ ]:
# Cell 5 - Hash-link final claim-package source, config and notebook.

HASH_TARGETS = [
    'configs/phase1/final_claim_package.json',
    'src/phase1/final_claim_package.py',
    'src/phase1/claim_state.py',
    'src/phase1/controls.py',
    'src/phase1/calibration.py',
    'src/phase1/influence.py',
    'src/cli.py',
    'tests/unit/test_phase1_final_claim_package.py',
    'configs/models/distill_a3.yaml',
    'configs/models/privileged_a4.yaml',
    'configs/controls/control_suite_spec.yaml',
    'configs/eval/claim_mapping.yaml',
    'configs/eval/metrics.yaml',
    'configs/eval/inference_defaults.yaml',
    'configs/gate1/decision_simulation.json',
    'configs/gate2/synthetic_validation.json',
    'notebooks/16_colab_phase1_final_claim_package_plan.ipynb',
]

hash_manifest = []
for rel in HASH_TARGETS:
    path = REPO_DIR / rel
    exists = path.exists()
    hash_manifest.append({'path': rel, 'exists': exists, 'sha256': sha256_file(path) if exists else None})
print(json.dumps(hash_manifest, indent=2))
assert all(item['exists'] for item in hash_manifest), 'All final claim-package source/config/notebook files must exist.'

## Expected Artifact Contract

The CLI command below writes a timestamped non-claim plan under `artifacts/phase1_final_claim_package_plan/`:

- `phase1_final_claim_package_plan_inputs.json`
- `phase1_final_claim_package_plan_summary.json`
- `phase1_final_claim_package_plan_report.md`
- `phase1_final_claim_package_contract.json`
- `phase1_final_comparator_readiness.json`
- `phase1_final_governance_boundary_review.json`
- `phase1_final_claim_blocker_inventory.json`
- `phase1_final_claim_state_plan.json`
- `phase1_final_implementation_plan.json`

Expected current interpretation: blocked/non-claim. Missing final comparator, control, calibration, influence and reporting artifacts must remain blockers.

In [ ]:
# Cell 6 - Run the CLI-backed final claim-package plan.
# This command records a contract and blockers only. It does not run Phase 1 training.

cmd = [
    'python', '-m', 'src.cli', 'phase1_final_claim_package_plan',
    '--profile', 't4_safe',
    '--config', str(PREREG_BUNDLE),
    '--governance-run', str(GOVERNANCE_READINESS_RUN),
    '--output-root', str(OUTPUT_ROOT),
    '--package-config', 'configs/phase1/final_claim_package.json',
]
run(cmd, cwd=REPO_DIR)
print('Final claim-package plan command completed. Review blockers before implementing any claim-bearing runner.')

In [ ]:
# Cell 7 - Verify plan artifacts and closeout state.

run_dir = latest_run(OUTPUT_ROOT)
print('Latest final claim-package plan output:', run_dir)
required_files = [
    'phase1_final_claim_package_plan_inputs.json',
    'phase1_final_claim_package_plan_summary.json',
    'phase1_final_claim_package_plan_report.md',
    'phase1_final_claim_package_contract.json',
    'phase1_final_comparator_readiness.json',
    'phase1_final_governance_boundary_review.json',
    'phase1_final_claim_blocker_inventory.json',
    'phase1_final_claim_state_plan.json',
    'phase1_final_implementation_plan.json',
]
for name in required_files:
    exists = (run_dir / name).exists()
    print(f'{name} exists = {exists}')
    assert exists, f'Missing required artifact: {name}'

summary = read_json(run_dir / 'phase1_final_claim_package_plan_summary.json')
contract = read_json(run_dir / 'phase1_final_claim_package_contract.json')
claim_state = read_json(run_dir / 'phase1_final_claim_state_plan.json')
blockers = read_json(run_dir / 'phase1_final_claim_blocker_inventory.json')

print(json.dumps({
    'run_dir': str(run_dir),
    'summary_status': summary.get('status'),
    'package_status': summary.get('package_status'),
    'claim_ready': summary.get('claim_ready'),
    'headline_phase1_claim_open': summary.get('headline_phase1_claim_open'),
    'full_phase1_claim_bearing_run_allowed': summary.get('full_phase1_claim_bearing_run_allowed'),
    'required_final_comparators': summary.get('required_final_comparators'),
    'locked_threshold_references': contract.get('locked_threshold_references'),
    'blockers': blockers.get('blockers'),
}, indent=2))

assert summary.get('status') == 'phase1_final_claim_package_plan_recorded'
assert summary.get('package_status') == 'planning_non_claim'
assert summary.get('claim_ready') is False
assert summary.get('headline_phase1_claim_open') is False
assert summary.get('full_phase1_claim_bearing_run_allowed') is False
assert claim_state.get('status') == 'phase1_final_claim_state_plan_blocked'
assert claim_state.get('claim_ready') is False
assert contract.get('claim_opening_rules', {}).get('smoke_metrics_allowed_as_claim_evidence') is False
for expected_blocker in [
    'final_claim_package_config_not_locked',
    'a3_final_comparator_not_ready',
    'a4_final_comparator_not_ready',
    'final_control_results_missing',
    'final_calibration_package_missing',
    'final_influence_package_missing',
    'final_reporting_package_missing',
]:
    assert expected_blocker in blockers.get('blockers', []), f'Missing blocker: {expected_blocker}'
print('\nNOT OK TO CLAIM: This final claim-package plan does not prove decoder efficacy, A3/A4 efficacy, A4 superiority, privileged-transfer efficacy, or full Phase 1 neural comparator performance.')

## Next Engineering Boundary

Allowed next implementation work: implement final comparator/control/calibration/influence/reporting artifacts against this contract, one package at a time, with tests and fail-closed claim-state checks.

Not allowed: changing `planning_non_claim` to a claim-ready status, marking A3/A4 final-ready, or using smoke metrics as evidence before final artifacts exist and pass locked thresholds.